Import necessary functions

In [2]:
import numpy as np
import pandas as pd
import os
import datetime
from lists import *

Functions to read in all the data and drop any unnecessary columns present in the data.

In [3]:
def read_data(directory_list):
    temp = pd.read_csv(directory_list[0])
    drop_list_0 = [feat_name for feat_name in temp.columns if 'Data Source'.lower() in feat_name.lower()
                 or 'PERIOD'.lower() in feat_name.lower() or 'Frequency'.lower() in feat_name.lower() or
                 'Deficit [mm]' in feat_name or 'Runoff [mm]' in feat_name or 'Grass Temperature [Deg C]' in feat_name]
    temp.drop(drop_list_0, axis=1, inplace=True)
    for directory in directory_list[1:]:
        weather_data = pd.read_csv(directory)
        drop_list = [feat_name for feat_name in weather_data.columns if 'Data Source'.lower() in feat_name.lower()
                     or 'PERIOD'.lower() in feat_name.lower() or 'Frequency'.lower() in feat_name.lower() or
                     'Deficit [mm]' in feat_name or 'Runoff [mm]' in feat_name or 'Grass Temperature [Deg C]' in feat_name
                     or 'Minimum Temperature' in feat_name or 'Maximum Temperature' in feat_name]
        weather_data.drop(drop_list, axis=1, inplace=True)
        temp = pd.merge(left=temp, right=weather_data, on='Observation time UTC', how='left')

    return temp

Add prefixes to columns from data, so we can tell which data comes from which station

In [4]:
def add_prefix(df, location):
    df = df.add_prefix(f"{location}_")
    df.rename(columns={f"{location}_Observation time UTC": "Observation time UTC"}, inplace=True)
    return df

Proceed to read in all the data.  The station locations were selected based on data availability, data quality and proximity to MOTAT.  By choosing stations which sat around MOTAT, it becomes easier for the model to predict incoming rain events.  The location_list variable, is a list which is present in the lists.py file.  Using list comprehension to generate column names again and again really slowed down the run time during development, so the lists were generated once and then read from when necessary.

In [6]:
motat_weather = add_prefix(read_data(motat_list), location_list[0])
whitianga_weather = add_prefix(read_data(whitianga_list), location_list[1])
dargaville_weather = add_prefix(read_data(dargaville_list), location_list[2])
mangere_weather = add_prefix(read_data(mangere_list), location_list[3])
albany_weather = add_prefix(read_data(albany_list), location_list[4])

/var/folders/mt/1768jphj4lvggkq0l_clwyjc0000gn/T/ipykernel_3383/2662723671.py:2: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  temp = pd.read_csv(directory_list[0])


Join all of the different station data into one DataFrame, and then write the DataFrame to disk as a csv file.  This keeps the loaded data in a safe place and avoids subsequent reloads.

In [9]:
pre_feat_eng_data = mangere_weather
for weather in [whitianga_weather, dargaville_weather, motat_weather, albany_weather]:
    pre_feat_eng_data = pd.merge(left=pre_feat_eng_data, right=weather,on='Observation time UTC', how='left')

path = 'pre_feat_eng_data.csv'

if os.path.exists(path=path):
    print('already written to disk')
else:
    pre_feat_eng_data.to_csv('pre_feat_eng_data.csv')
    print("written to disk")

already written to disk


Sort the data timewise.  This is a time series analysis so not doing this would cause a massive problem, especially during the machine learning phase where data leakage would falsify model performance.

In [10]:
pre_feat_eng_data = pre_feat_eng_data.sort_values(by='Observation time UTC')


Set up the sentinel value function.  Sentinel filling is where NaN values are replaced by other values which lay far outside the domain of sensible values for a certain column.  For example, rainfall can't be negative so it has its NaN values replaced by something which is negative and massive.  This can tell the our ML algorithms that there's no useful information in those cells of tabular data.
Some ML algorithms like XGBoost from teh xgboost library can ignore NaNs, while other algorithms such as RandomForest from scikit-learn can't ignore NaNs, and require the implementation of sentinel values.

In [11]:
def sentinel_fill(lag_list, lag_df_shape, lag_df_cols, matching_index=None):
    feat_keys = {"Labels":[-150., -80.],"Humidity":[1e7, 1.2e7], "Speed":[-69420.,-42069.], "Pressure":[-20000., -10000.], "Max Temp":[-2000., -1200.],
                 "Min Temp":[1200, 2000], "Mean Temp":[110000., 119000.], "Direction":[-1e9,-1e8], "Rain":[-800.,-700.]}
    fill_array = np.ones(lag_df_shape)
    rng = np.random.default_rng(69420)
    fill_vals = rng.uniform(low=feat_keys[lag_list[0]][0], high=feat_keys[lag_list[0]][1],size=(1,lag_df_shape[1]))
    if matching_index is not None:
        fill_df = pd.DataFrame(data=fill_array * fill_vals, columns=list(lag_df_cols), index=matching_index)
    else:
        fill_df = pd.DataFrame(data=fill_array * fill_vals, columns=list(lag_df_cols))
    return fill_df

Use the sentinel fill function to replace any of the NaNs in the